In [1]:
import pandas as pd

In [60]:
import numpy as np

In [2]:
df = pd.read_csv("Heart Prediction Quantum Dataset.csv")

In [3]:
df.head()

,Age,Gender,BloodPressure,Cholesterol,HeartRate,HeartDisease
0,68,Hombre,105.00,191,107,1
1,58,Mujer,97.00,249,89,0
2,44,Mujer,93.00,190,82,1
3,72,Hombre,93.00,183,101,1
4,37,Mujer,145.00,166,103,1


In [4]:
df.shape

(500, 6)

In [5]:
df.describe()

,Age,Cholesterol,HeartRate,HeartDisease
count,500.000000,500.00000,500.000000,500.000000
mean,54.864000,221.50000,88.766000,0.600000
std,14.315004,43.86363,17.417289,0.490389
min,30.000000,150.00000,60.000000,0.000000
25%,43.000000,183.75000,73.000000,0.000000
50%,55.000000,221.00000,89.000000,1.000000
75%,66.250000,258.00000,104.000000,1.000000
max,79.000000,299.00000,119.000000,1.000000


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Age            500 non-null    int64 
 1   Gender         500 non-null    object
 2   BloodPressure  500 non-null    object
 3   Cholesterol    500 non-null    int64 
 4   HeartRate      500 non-null    int64 
 5   HeartDisease   500 non-null    int64 
dtypes: int64(4), object(2)
memory usage: 23.6+ KB


### Limpieza

In [7]:
df["BloodPressure"] = df["BloodPressure"].str.replace(',', '.')
df["BloodPressure"] = df["BloodPressure"].astype(float)

In [8]:
gender_lower = df['Gender'].str.lower()
gender_clean = gender_lower.str.rstrip(" {}")
gender_clean[gender_clean == 'ombre'] = 'hombre'
df['Gender'] = gender_clean.replace({'hombre': 1, 'mujer': 0})

gender_clean.unique()

C:\Users\ceciq\AppData\Local\Temp\ipykernel_22396\654190138.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Gender'] = gender_clean.replace({'hombre': 1, 'mujer': 0})


array(['hombre', 'mujer'], dtype=object)

In [9]:
df = df.drop_duplicates()

In [10]:
df.head()

,Age,Gender,BloodPressure,Cholesterol,HeartRate,HeartDisease
0,68,1,105.0,191,107,1
1,58,0,97.0,249,89,0
2,44,0,93.0,190,82,1
3,72,1,93.0,183,101,1
4,37,0,145.0,166,103,1


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Age            500 non-null    int64  
 1   Gender         500 non-null    int64  
 2   BloodPressure  500 non-null    float64
 3   Cholesterol    500 non-null    int64  
 4   HeartRate      500 non-null    int64  
 5   HeartDisease   500 non-null    int64  
dtypes: float64(1), int64(5)
memory usage: 23.6 KB


In [12]:
df.columns

Index(['Age', 'Gender', 'BloodPressure', 'Cholesterol', 'HeartRate',
       'HeartDisease'],
      dtype='object')

### Regresión

In [63]:
from sklearn.linear_model import LogisticRegression

X = df[['Age', 'Gender', 'BloodPressure', 'Cholesterol', 'HeartRate']]
y = df['HeartDisease']

model = LogisticRegression()
model.fit(X, y)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [64]:
model.intercept_

array([7.06951308])

In [65]:
model.coef_

array([[ 0.05036482, -0.02863074, -0.01585793, -0.0251093 , -0.01787389]])

Con C = 1e-6

In [70]:
model_2 = LogisticRegression(C=1e-6)
model_2.fit(X, y)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1e-06
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [71]:
model_2.intercept_

array([1.31174549])

In [72]:
model_2.coef_

array([[ 8.80812814e-04, -1.19458732e-06, -7.18407429e-04,
        -3.69815601e-03, -4.21058210e-04]])

Con C = 1e6

In [73]:
model_3 = LogisticRegression(C=1e6)
model_3.fit(X, y)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1000000.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [74]:
model_3.intercept_

array([7.07049453])

In [75]:
model_3.coef_

array([[ 0.05036867, -0.02996471, -0.01585998, -0.02510933, -0.01787689]])

### Matriz de confusión

In [66]:
y_proba = model.predict_proba(X)[:, 1]

**Threshold = 0.3**

In [76]:
y_pred_03 = (y_proba >= 0.3)

TP_03 = np.sum((y_pred_03 == 1) & (y == 1))
TN_03 = np.sum((y_pred_03 == 0) & (y == 0))
FP_03 = np.sum((y_pred_03 == 1) & (y == 0))
FN_03 = np.sum((y_pred_03 == 0) & (y == 1))

print(TP_03, TN_03, FP_03, FN_03)

281 73 127 19


In [81]:
accuracy_03 = (TP_03 + TN_03) / (TP_03 + FP_03 + TN_03 + FN_03)
print('Accuracy = ', accuracy_03)

Accuracy =  0.708


In [82]:
recall_03 = TP_03 / (TP_03 + FN_03)
print('Recall = ', recall_03)

Recall =  0.9366666666666666


In [83]:
precision_03 = TP_03 / (TP_03 + FP_03)
print('Precisión = ', precision_03)

Precisión =  0.6887254901960784


In [84]:
specificity_03 = TN_03 / (TN_03 + FP_03)
print('Specificity = ', specificity_03)

Specificity =  0.365


**Treshold = 0.5**

In [77]:
y_pred_05 = (y_proba >= 0.5)

TP_05 = np.sum((y_pred_05 == 1) & (y == 1))
TN_05 = np.sum((y_pred_05 == 0) & (y == 0))
FP_05 = np.sum((y_pred_05 == 1) & (y == 0))
FN_05 = np.sum((y_pred_05 == 0) & (y == 1))

print(TP_05, TN_05, FP_05, FN_05)

238 127 73 62


In [85]:
accuracy_05 = (TP_05 + TN_05) / (TP_05 + FP_05 + TN_05 + FN_05)
print('Accuracy = ', accuracy_05)

Accuracy =  0.73


In [87]:
recall_05 = TP_05 / (TP_05 + FN_05)
print('Recall = ', recall_05)

Recall =  0.7933333333333333


In [89]:
precision_05 = TP_05 / (TP_05 + FP_05)
print('Precisión = ', precision_05)

Precisión =  0.7652733118971061


In [91]:
specificity_05 = TN_05 / (TN_05 + FP_05)
print('Specificity = ', specificity_05)

Specificity =  0.635


**Treshold = 0.8**

In [78]:
y_pred_08 = (y_proba >= 0.8)

TP_08 = np.sum((y_pred_08 == 1) & (y == 1))
TN_08 = np.sum((y_pred_08 == 0) & (y == 0))
FP_08 = np.sum((y_pred_08 == 1) & (y == 0))
FN_08 = np.sum((y_pred_08 == 0) & (y == 1))

print(TP_08, TN_08, FP_08, FN_08)

134 184 16 166


In [86]:
accuracy_08 = (TP_08 + TN_08) / (TP_08 + FP_08 + TN_08 + FN_08)
print('Accuracy = ', accuracy_08)

Accuracy =  0.636


In [88]:
recall_08 = TP_08 / (TP_08 + FN_08)
print('Recall = ', recall_08)

Recall =  0.44666666666666666


In [90]:
precision_08 = TP_08 / (TP_08 + FP_08)
print('Precisión = ', precision_08)

Precisión =  0.8933333333333333


In [92]:
specificity_08 = TN_08 / (TN_08 + FP_08)
print('Specificity = ', specificity_08)

Specificity =  0.92


### **Interpretación**

Al subir el umbral, el modelo se vuelve más estricto para detectar enfermos. Con 0.3 detecta a casi todos, pero genera muchas falsas alarmas. Con 0.8 hay menos falsas alarmas, pero se escapan muchos enfermos.

El umbral 0.5 tiene el mejor equilibrio, con un accuracy de 0.73. Por lo tanto, un umbral bajo sirve si se busca detectar la mayor cantidad de enfermos, mientras que uno alto sirve si se quieren evitar falsas alarmas.